# **`Big Data Analytics`**


---
1. Import Libraries

2. Create Spark Session
3. Load Datasets
4. Data Exploration
5. Data Cleaning
6. Merge Tables
7. Big Data Analysis
    *   Total Revenue
    
    *   Monthly Sales
    *   Top Categories
    *   Top Customers
    *   Payment Analysis
    *   Delivery Analysis
    *   Top States












In [1]:
# !pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder \
    .appName("Olist Big Data Analysis") \
    .getOrCreate()

C:\Users\gopin\AppData\Roaming\Python\Python313\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [ ]:
orders = spark.read.csv("Datasets/olist_orders_dataset.csv", header=True, inferSchema=True)

customers = spark.read.csv("Datasets/olist_customers_dataset.csv", header=True, inferSchema=True)

order_items = spark.read.csv("Datasets/olist_order_items_dataset.csv", header=True, inferSchema=True)

payments = spark.read.csv("Datasets/olist_order_payments_dataset.csv", header=True, inferSchema=True)

products = spark.read.csv("Datasets/olist_products_dataset.csv", header=True, inferSchema=True)

translation = spark.read.csv("Datasets/product_category_name_translation.csv", header=True,)

In [5]:
orders.printSchema()

customers.printSchema()

order_items.printSchema()

payments.printSchema()

products.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |

In [6]:
# Checking Null Values

for df, name in [
    (orders,"Orders"),
    (customers,"Customers"),
    (order_items,"Order Items"),
    (payments,"Payments"),
    (products,"Products")
]:
    print("\n",name)
    df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()


 Orders
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+


 Customers
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_p

In [7]:
weight = products.select(avg("product_weight_g")).first()[0]
length = products.select(avg("product_length_cm")).first()[0]
height = products.select(avg("product_height_cm")).first()[0]
width = products.select(avg("product_width_cm")).first()[0]

products = products.fillna({
    "product_weight_g": weight,
    "product_length_cm": length,
    "product_height_cm": height,
    "product_width_cm": width
})

products = products.fillna({
    "product_category_name": "Unknown"
})

payments = payments.na.drop(subset=["payment_value"])

customers = customers.na.drop()

order_items = order_items.na.drop(
    subset=["price", "freight_value"]
)


In [8]:
# Rechecking Null Values

for df, name in [
    (orders,"Orders"),
    (customers,"Customers"),
    (order_items,"Order Items"),
    (payments,"Payments"),
    (products,"Products")
]:
    print("\n",name)
    df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()


 Orders
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+


 Customers
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_p

In [9]:
# Removing Duplicate Values

orders = orders.dropDuplicates()

customers = customers.dropDuplicates()

order_items = order_items.dropDuplicates()

payments = payments.dropDuplicates()

products = products.dropDuplicates()

In [10]:
df = orders.join(customers,"customer_id","left") \
           .join(order_items,"order_id","left") \
           .join(payments,"order_id","left") \
           .join(products,"product_id","left")


In [11]:
df = df.join(
    translation,
    "product_category_name",
    "left"
)

In [12]:
df.select(
    round(sum("payment_value") / 1000000, 2).alias("Revenue (Million BRL)")
).show()

+---------------------+
|Revenue (Million BRL)|
+---------------------+
|                20.47|
+---------------------+



In [13]:
monthly_sales = df.withColumn(
    "Month",
    date_format("order_purchase_timestamp", "yyyy-MM")
)

monthly_sales.groupBy("Month") \
    .agg(
        round(sum("payment_value") / 1000000, 2).alias("Revenue (Million BRL)")
    ) \
    .orderBy("Month") \
    .show(50, False)

+-------+---------------------+
|Month  |Revenue (Million BRL)|
+-------+---------------------+
|2016-09|0.0                  |
|2016-10|0.08                 |
|2016-12|0.0                  |
|2017-01|0.19                 |
|2017-02|0.35                 |
|2017-03|0.54                 |
|2017-04|0.51                 |
|2017-05|0.73                 |
|2017-06|0.61                 |
|2017-07|0.74                 |
|2017-08|0.88                 |
|2017-09|1.02                 |
|2017-10|1.03                 |
|2017-11|1.6                  |
|2017-12|1.06                 |
|2018-01|1.42                 |
|2018-02|1.31                 |
|2018-03|1.48                 |
|2018-04|1.5                  |
|2018-05|1.51                 |
|2018-06|1.3                  |
|2018-07|1.36                 |
|2018-08|1.25                 |
|2018-09|0.0                  |
|2018-10|0.0                  |
+-------+---------------------+



In [14]:
df.groupBy("product_category_name_english") \
    .agg(
        round(sum("payment_value") / 1000000, 4)
        .alias("Revenue (Million BRL)")
    ) \
    .orderBy(desc("Revenue (Million BRL)")) \
    .show(10, False)

+-----------------------------+---------------------+
|product_category_name_english|Revenue (Million BRL)|
+-----------------------------+---------------------+
|bed_bath_table               |1.7126               |
|health_beauty                |1.6574               |
|computers_accessories        |1.5853               |
|furniture_decor              |1.4302               |
|watches_gifts                |1.4292               |
|sports_leisure               |1.3921               |
|housewares                   |1.0948               |
|auto                         |0.8523               |
|garden_tools                 |0.8383               |
|cool_stuff                   |0.7797               |
+-----------------------------+---------------------+
only showing top 10 rows


In [15]:
df.groupBy("payment_type") \
    .agg(
        count("*").alias("Orders"),
        round(sum("payment_value") / 1000000, 4)
        .alias("Revenue (Million BRL)")
    ) \
    .orderBy(desc("Revenue (Million BRL)")) \
    .show(10,False)

+------------+------+---------------------+
|payment_type|Orders|Revenue (Million BRL)|
+------------+------+---------------------+
|credit_card |87286 |15.6949              |
|boleto      |23037 |4.0868               |
|voucher     |6407  |0.4326               |
|debit_card  |1698  |0.2564               |
|not_defined |3     |0.0                  |
|NULL        |3     |NULL                 |
+------------+------+---------------------+



In [16]:
delivery = df.withColumn(
    "Delivery Days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_purchase_timestamp")
    )
)

delivery.select(
    round(avg("Delivery Days"),0).alias("Average Delivery Days")
).show()

+---------------------+
|Average Delivery Days|
+---------------------+
|                 12.0|
+---------------------+



In [17]:
late = delivery.withColumn(
    "Late",
    when(
        col("order_delivered_customer_date") >
        col("order_estimated_delivery_date"),
        1
    ).otherwise(0)
)

late.groupBy("Late").count().show()

+----+------+
|Late| count|
+----+------+
|   1|  9027|
|   0|109407|
+----+------+



In [18]:
df.toPandas().to_csv("olist_cleaned.csv", index=False)

C:\Users\gopin\AppData\Roaming\Python\Python313\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\gopin\AppData\Roaming\Python\Python313\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
C:\Users\gopin\AppData\Roaming\Python\Python313\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
